In [1]:
# Setup: carica la struttura e definisce i parametri usati nel notebook

import sys
from pathlib import Path


def _setup_path():
    root = Path.cwd()
    for _ in range(10):
        if (root / "pyproject.toml").exists():
            break
        root = root.parent
    src_path = str(root / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    return root


PROJECT_ROOT = _setup_path()

import numpy as np

from computational_course.ovito_visualization import load_structure

FILENAME = "fcc100a108.txt"
_txt_path, pos = load_structure(FILENAME, PROJECT_ROOT)
natoms = pos.shape[0]

# Parametri Lennard-Jones per Ag
sigma = 2.644
epsilon = 0.345
eps = epsilon

print(f"File: {_txt_path.name}")
print(f"Atomi: {natoms}")
print(f"Shape pos: {pos.shape}")

File: fcc100a108.txt
Atomi: 108
Shape pos: (108, 3)


# 9 Ottobre 

## Continuazione NN, primi secondi e terzi vicini. L-J cutoff ecc

### scriviamo ora un codice per migliorare la velocità computazionale che fino ad ora includeva un calcolo nidificato dell'ordine del numero di atomi disponibili al quadrato. voglio sfruttare la condizione dei  vicini (NBRS) e del raggio di cutoff del potenziale (energia Epot) in modo da renderlo scalabile linearmente.

In [2]:
# quanti vicini?

import numpy as np

R = 4.6  # Raggio di vicinato in Å

natoms = pos.shape[0]
nnbrs = np.zeros(natoms, dtype=int)

for i in range(natoms):
    for j in range(natoms):
        if i != j:
            dx = pos[i, 0] - pos[j, 0]
            dy = pos[i, 1] - pos[j, 1]
            dz = pos[i, 2] - pos[j, 2]

            distance = np.sqrt(dx**2 + dy**2 + dz**2)

            if distance < R:
                nnbrs[i] += 1

print("Numero di atomi:", natoms)
print("Numero di vicini per ciascun atomo:", nnbrs)

Numero di atomi: 108
Numero di vicini per ciascun atomo: [11  6  9 12  8  8  9 12 13 13 12  9  8  8 12  9 11  6 17 17 15  8 16 12
 16 12  8 15 12 16 12 16 11 11 11 11 12 12 16  9 12 12 18 18 13 17 17 13
 13 17 17 13 16  9  9 16 17 13 12 12 17 13 13 17 16  9 18 18 13 17 12 12
 16 12 17 17 12 16 16 12 11 11 15  8 11 11 12 16 15  8  9 12 11  6  9 12
 13 13  6 11 12  9 12  9  8  8  8  8]


In [3]:
# quale vicino?

R = 4.6
atom_i = 1

vicini = []

for j in range(natoms):
    if j != atom_i:
        dx = pos[atom_i, 0] - pos[j, 0]
        dy = pos[atom_i, 1] - pos[j, 1]
        dz = pos[atom_i, 2] - pos[j, 2]

        distance = np.sqrt(dx**2 + dy**2 + dz**2)

        if distance < R:
            vicini.append(j)

print(f"Atomo selezionato: {atom_i}")
print(f"Numero di vicini: {len(vicini)}")
print(f"Indici dei vicini: {vicini}")

Atomo selezionato: 1
Numero di vicini: 6
Indici dei vicini: [0, 2, 6, 32, 33, 39]


In [4]:
# wich NBRG of atom[12(1),0] is = 4?
# --- atom[12(2),0] is =

# wich NBRS of atom[i, NNBRS[i]-1] = j?


# Inizializzazione
force = np.zeros((natoms, 3))
epot = 0.0
nnbrs = np.zeros(natoms, dtype=int)
whichnbr = np.zeros((natoms, natoms), dtype=int)  # questo è DA CAPIRE.

rc = 3

# 1. Calcolo dei vicini entro il cutoff rc
for i in range(natoms):
    for j in range(natoms):
        if i != j:
            rij = pos[i] - pos[j]
            dist = np.linalg.norm(rij)

            if dist < rc:
                whichnbr[i, nnbrs[i]] = j  # QUESTO è DA CAPIRE
                nnbrs[i] += 1


# 2. Calcolo dell'energia potenziale considerando, per OGNI ATOMO,
# esclusivamente le interazioni con i suoi vicini già individuati
for i in range(natoms):
    for k in range(nnbrs[i]):
        j = whichnbr[i, k]  # DA CAPIRE

        rij = pos[i] - pos[j]
        dist = np.linalg.norm(rij)

        sr6 = (sigma / dist) ** 6
        phi = 4.0 * epsilon * (sr6**2 - sr6)

        # Dividiamo per 2 perché ogni coppia è contata due volte
        epot += 0.5 * phi


# Risultati

print(nnbrs)
print("\n")
print(whichnbr)
print("\n")
print(f"Numero di atomi                 : {natoms}")
print(f"Energia potenziale totale       : {epot:.6f} eV")
print(f"Numero medio di vicini          : {np.mean(nnbrs):.2f}")

[ 8  3  5  8  5  5  5  8  8  8  8  5  5  5  8  5  8  3 12 12 12  5 12  8
 12  8  5 12  8 12  8 12  8  8  8  8  8  8 12  5  8  8 12 12  8 12 12  8
  8 12 12  8 12  5  5 12 12  8  8  8 12  8  8 12 12  5 12 12  8 12  8  8
 12  8 12 12  8 12 12  8  8  8 12  5  8  8  8 12 12  5  5  8  8  3  5  8
  8  8  3  8  8  5  8  5  5  5  5  5]


[[ 1  2  6 ...  0  0  0]
 [ 0 32 33 ...  0  0  0]
 [ 0  3 24 ...  0  0  0]
 ...
 [76 82 83 ...  0  0  0]
 [79 88 89 ...  0  0  0]
 [73 88 89 ...  0  0  0]]


Numero di atomi                 : 108
Energia potenziale totale       : -154.795762 eV
Numero medio di vicini          : 8.33


# QUINDI IMPORTANTE:

nnbrs è un array che registra quanti vicini ha ciascun atomo, mentre whichnbr è una matrice che memorizza gli indici degli atomi vicini. Ogni riga di whichnbr corrisponde a un atomo e ogni elemento della riga contiene l'indice di un suo vicino. Inizialmente entrambe le strutture sono impostate a zero; quando viene trovato un atomo a distanza inferiore al cutoff, il suo indice viene salvato in whichnbr e il contatore nnbrs viene incrementato.

Gli zeri rimanenti in ogni riga sono semplicemente valori iniziali non utilizzati: per sapere quali elementi sono validi, si guarda nnbrs[i]



-----


Questo approccio serve a ridurre il numero di interazioni da calcolare e rendere più efficiente la simulazione atomistica.
Primo ciclo: confronti ogni atomo con tutti gli altri per identificare quelli che si trovano entro il cutoff rc.

Secondo ciclo: calcoli l'energia potenziale considerando soltanto le coppie di atomi vicini già individuate, senza ripetere il confronto con tutti gli atomi del sistema.

Vantaggio: per sistemi con molti atomi, questo metodo può ridurre notevolmente il costo computazionale, soprattutto se ogni atomo ha pochi vicini rispetto al numero totale di atomi.

In sostanza, separi la ricerca dei vicini dal calcolo delle interazioni. Questo è utile perché il potenziale di Lennard-Jones dipende dalla distanza tra le coppie di atomi e, con un cutoff, puoi trascurare le interazioni oltre la distanza scelta.
Nota: nel tuo codice attuale, la ricerca iniziale confronta ancora tutte le coppie di atomi; il vantaggio si ottiene soprattutto quando riutilizzi la lista dei vicini per calcolare energia e forze, senza ricostruirla ogni volta.


----


Nel mio codice, con rc = 3.0 Å:
Se la distanza tra due atomi è minore di 3.0 Å, vengono considerati vicini e la loro interazione contribuisce all'energia potenziale.
Se la distanza è maggiore o uguale a 3.0 Å, quella coppia NON viene inserita nella lista dei vicini e la sua INTERAZIONE VIENE IGNORATA.
Esempio: l'atomo 0 può avere due vicini entro il cutoff e altri tre atomi più lontani. L'atomo 0 rimane nel sistema, ma soltanto le interazioni con i due vicini vengono considerate nel calcolo.

In [5]:
# dinamical double loop:

# Code which computes nbrs, then computes forces based on nbrs, then epot based on nbrs (1/3)

natoms = pos.shape[0]
force = np.zeros((natoms, 3))
nnbrs = np.zeros(natoms, int)
whichnbr = np.zeros((natoms, natoms), int)
rc = 3.0
eps = 0.345
sigma = 2.644
# 1) I compute the nbrs
for i in range(0, natoms):
    for j in range(0, natoms):
        # K = whichnbr[i,j]
        dx = pos[i, 0] - pos[j, 0]
        dy = pos[i, 1] - pos[j, 1]
        dz = pos[i, 2] - pos[j, 2]
        d2 = dx * dx + dy * dy + dz * dz
        dis = d2**0.5
        if dis < (rc) and i != j:
            nnbrs[i] = nnbrs[i] + 1
            whichnbr[i, nnbrs[i] - 1] = j
# DISTANCE BETWEEN ATOM AND THE NBRS OF ANTOHER ATHOM

In [6]:
# dinamical double loop:

natoms = pos.shape[0]
force = np.zeros((natoms, 3))
nnbrs = np.zeros(natoms, int)
whichnbr = np.zeros((natoms, natoms), int)
rc = 3.0
eps = 0.345
sigma = 2.644

for i in range(0, natoms):
    for j in range(0, natoms):
        k = whichnbr[i, j]
        dx = pos[i, 0] - pos[k, 0]
        dy = pos[i, 1] - pos[k, 1]
        dz = pos[i, 2] - pos[k, 2]
        d2 = dx * dx + dy * dy + dz * dz
        dis = d2**0.5
        epot = ...  # lennard jones

# DISTANCE BETWEEN ATOM AND THE NBRS OF ANTOHER ATHOM

In [7]:
# codice corretto:


natoms = pos.shape[0]
force = np.zeros((natoms, 3))
nnbrs = np.zeros(natoms, int)
whichnbr = np.zeros((natoms, natoms), int)

rc = 4.0
eps = 0.345
sigma = 2.644

# 1. Calcolo dei vicini
for i in range(natoms):
    for j in range(natoms):
        if i != j:
            dx = pos[i, 0] - pos[j, 0]
            dy = pos[i, 1] - pos[j, 1]
            dz = pos[i, 2] - pos[j, 2]

            d2 = dx**2 + dy**2 + dz**2
            dis = d2**0.5

            if dis < rc:
                whichnbr[i, nnbrs[i]] = j
                nnbrs[i] += 1

# 2. Calcolo dell'energia potenziale usando i vicini
epot = 0.0

for i in range(natoms):
    for j in range(nnbrs[i]):
        k = whichnbr[i, j]

        dx = pos[i, 0] - pos[k, 0]
        dy = pos[i, 1] - pos[k, 1]
        dz = pos[i, 2] - pos[k, 2]

        d2 = dx**2 + dy**2 + dz**2
        dis = d2**0.5

        sr6 = (sigma / dis) ** 6
        phi = 4 * eps * (sr6**2 - sr6)

        epot += 0.5 * phi

print("Energia potenziale totale:", epot, "eV")
print("Numero medio di vicini:", np.mean(nnbrs))


Energia potenziale totale: -154.7957616328464 eV
Numero medio di vicini: 8.333333333333334


# Legenda: lista dei vicini e energia di Lennard-Jones

## 1. Variabili

| Nome | Tipo / forma | Significato |
|---|---|---|
| `natoms` | int (108) | numero di atomi |
| `pos` | array `(natoms, 3)` | coordinate x, y, z di ogni atomo |
| `sigma`, `epsilon` | float | parametri di Lennard-Jones (distanza e profondità della buca) |
| `rc` | float (3.0) | raggio di cutoff: oltre questa distanza due atomi non sono vicini |
| `nnbrs` | array `(natoms,)` | `nnbrs[i]` = **quanti** vicini ha l'atomo `i` |
| `whichnbr` | array `(natoms, natoms)` | `whichnbr[i, k]` = **indice** del k-esimo vicino dell'atomo `i` |
| `epot` | float | energia potenziale totale accumulata |
| `force` | array `(natoms, 3)` | inizializzata ma **non usata** in questo frammento |

## 2. Idea chiave: `nnbrs[i]` fa due lavori

```python
whichnbr[i, nnbrs[i]] = j   # scrive j nel prossimo slot libero della riga i
nnbrs[i] += 1               # sposta il contatore allo slot successivo
```

`nnbrs[i]` è insieme il **contatore** dei vicini trovati e l'**indice del prossimo slot libero**.
A fine ciclo coincide con il numero totale di vicini.

## 3. Come leggere `whichnbr`

- Una **riga per atomo**, una **colonna per vicino**.
- Sono validi **solo** i primi `nnbrs[i]` elementi di ogni riga:

```python
whichnbr[i, :nnbrs[i]]      # vicini veri dell'atomo i
whichnbr[i, nnbrs[i]-1]     # ultimo vicino valido
```

- Gli `0 0 0` a destra sono **riempimento** (da `np.zeros`), non "vicino = atomo 0".
- Esempio: riga `[0 32 33 ... 0 0 0]` con `nnbrs = 3` significa che i vicini sono 0, 32, 33.

## 4. Passi dell'algoritmo

1. **Costruzione della lista:** per ogni coppia `(i, j)` con `i != j` calcola
   $r_{ij} = |\vec r_i - \vec r_j|$; se `dist < rc`, aggiungi `j` ai vicini di `i`.
2. **Energia:** per ogni atomo `i` e per ogni suo vicino `j = whichnbr[i, k]`, `k = 0 … nnbrs[i]-1`:

$$\varphi(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right]$$

   Nel codice `sr6 = (sigma/dist)**6`, quindi `sr6**2` è il termine a potenza 12.
3. **Fattore 0.5:** ogni coppia $(i,j)$ compare due volte (in riga `i` e in riga `j`), quindi si somma $\tfrac12 \varphi$.

$$E_{pot} = \sum_i \sum_{j \in \text{vicini}(i)} \tfrac12\,\varphi(r_{ij})$$

## 5. Come leggere l'output

| Stampa | Cosa è |
|---|---|
| matrice grande | `whichnbr` (108 × 108, abbreviata con `...`) |
| vettore di interi | `nnbrs`: numero di vicini di ciascun atomo |
| `Numero di atomi` | `natoms` |
| `Energia potenziale totale` | `epot` in eV |
| `Numero medio di vicini` | `np.mean(nnbrs)`, la coordinazione media |

## 6. Controlli di coerenza

- **Simmetria:** se `j` è nella riga di `i`, allora `i` è nella riga di `j`.
- **Doppio conteggio:** `nnbrs.sum() / 2` = numero di coppie distinte (con 8.33 di media su 108 atomi, circa 450).
- **Coordinazione:** valori alti (es. 12) indicano atomi interni, valori bassi (3, 5, 8) atomi di superficie, spigolo o vertice.

## 7. Limiti del codice

- Costo $O(N^2)$: scorre tutte le coppie. Va bene per 108 atomi, non per sistemi grandi.
- Nessuna **condizione periodica al contorno**: le distanze sono quelle dirette, senza convenzione dell'immagine minima.
- Il cutoff è **troncato** (nessun shift dell'energia a `rc`), quindi `epot` dipende un po' dalla scelta di `rc`.

### NELLA DIMANICA, I VICINI CAMBIANO.


In [8]:
import numpy as np
import pandas as pd

natoms = pos.shape[0]
rc = 3.0  # sigma ed epsilon restano quelli già definiti

# 1. Distanze tra tutte le coppie (matrice natoms x natoms)
diff = pos[:, None, :] - pos[None, :, :]
dist = np.linalg.norm(diff, axis=2)

# 2. Tabella delle coppie entro il cutoff, escludendo i == j
i_idx, j_idx = np.where((dist < rc) & ~np.eye(natoms, dtype=bool))
pairs = pd.DataFrame({"i": i_idx, "j": j_idx, "dist": dist[i_idx, j_idx]})

# 3. Energia di Lennard-Jones per ogni coppia
sr6 = (sigma / pairs["dist"]) ** 6
pairs["phi"] = 4.0 * epsilon * (sr6**2 - sr6)

# Ogni coppia compare due volte (i->j e j->i), quindi 0.5
epot = 0.5 * pairs["phi"].sum()

# 4. nnbrs: numero di vicini per atomo (0 se l'atomo è isolato)
nnbrs = pairs.groupby("i").size().reindex(range(natoms), fill_value=0)

# 5. whichnbr: una riga per atomo, colonna k = k-esimo vicino
pairs["k"] = pairs.groupby("i").cumcount()
whichnbr = (
    pairs.pivot(index="i", columns="k", values="j")
    .reindex(range(natoms))
    .astype("Int64")
)  # <NA> dove non c'è vicino
whichnbr.insert(0, "nnbrs", nnbrs)
whichnbr.index.name = "atomo"

# Risultati
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(whichnbr)

print(f"Numero di atomi            : {natoms}")
print(f"Energia potenziale totale  : {epot:.6f} eV")
print(f"Numero medio di vicini     : {nnbrs.mean():.2f}")


# quelli che hanno pochi VICINI sono quelli di solito ai lati del cubo e se hanno pochi vicini danno un contributo alla interazione minore rispetto agli altri . in dinamica vedremo che quelli che hanno pochi vicini si muoveranno tanto e a volte si staccheranno dal cubo. quelli invece nel bulk (con 12 primi vicini ad esempio) si muovono poco.

k,nnbrs,0,1,2,3,4,5,6,7,8,9,10,11
atomo,,,,,,,,,,,,,
0,8,1,2,6,8,24,29,32,33,<NA>,<NA>,<NA>,<NA>
1,3,0,32,33,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,5,0,3,24,25,32,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,8,2,4,8,10,19,20,24,25,<NA>,<NA>,<NA>,<NA>
4,5,3,5,20,21,25,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,5,4,10,20,21,23,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,5,0,7,28,29,33,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,8,6,8,12,14,18,27,28,29,<NA>,<NA>,<NA>,<NA>
8,8,0,3,7,9,18,19,24,29,<NA>,<NA>,<NA>,<NA>


Numero di atomi            : 108
Energia potenziale totale  : -154.795762 eV
Numero medio di vicini     : 8.33
